In [157]:
import pandas as pd
import numpy as np
# Load the dataset 
df = pd.read_csv("../datasets/raw/hotel_bookings.csv")

In [158]:
df.head()

,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,children,babies,meal,country,market_segment,distribution_channel,is_repeated_guest,previous_cancellations,previous_bookings_not_canceled,reserved_room_type,assigned_room_type,booking_changes,deposit_type,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date
0,Resort Hotel,0,342,2015,July,27,1,0,0,2,0.0,0,BB,PRT,Direct,Direct,0,0,0,C,C,3,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,7/1/2015
1,Resort Hotel,0,737,2015,July,27,1,0,0,2,0.0,0,BB,PRT,Direct,Direct,0,0,0,C,C,4,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,7/1/2015
2,Resort Hotel,0,7,2015,July,27,1,0,1,1,0.0,0,BB,GBR,Direct,Direct,0,0,0,A,C,0,No Deposit,NaN,NaN,0,Transient,75.0,0,0,Check-Out,7/2/2015
3,Resort Hotel,0,13,2015,July,27,1,0,1,1,0.0,0,BB,GBR,Corporate,Corporate,0,0,0,A,A,0,No Deposit,304.0,NaN,0,Transient,75.0,0,0,Check-Out,7/2/2015
4,Resort Hotel,0,14,2015,July,27,1,0,2,2,0.0,0,BB,GBR,Online TA,TA/TO,0,0,0,A,A,0,No Deposit,240.0,NaN,0,Transient,98.0,0,1,Check-Out,7/3/2015


In [159]:
df.shape

(119390, 32)

In [160]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 119390 entries, 0 to 119389
Data columns (total 32 columns):
 #   Column                          Non-Null Count   Dtype  
---  ------                          --------------   -----  
 0   hotel                           119390 non-null  str    
 1   is_canceled                     119390 non-null  int64  
 2   lead_time                       119390 non-null  int64  
 3   arrival_date_year               119390 non-null  int64  
 4   arrival_date_month              119390 non-null  str    
 5   arrival_date_week_number        119390 non-null  int64  
 6   arrival_date_day_of_month       119390 non-null  int64  
 7   stays_in_weekend_nights         119390 non-null  int64  
 8   stays_in_week_nights            119390 non-null  int64  
 9   adults                          119390 non-null  int64  
 10  children                        119386 non-null  float64
 11  babies                          119390 non-null  int64  
 12  meal                       

In [161]:
# Convert reservation status date to datetime
df["reservation_status_date"] = pd.to_datetime(
    df["reservation_status_date"],
    errors="coerce"
)

# Check whether date conversion created missing values
df["reservation_status_date"].isna().sum()

np.int64(0)

In [162]:
# Summary statistics for all variables
df.describe(include="all").T

,count,unique,top,freq,mean,min,25%,50%,75%,max,std
hotel,119390,2,City Hotel,79330,NaN,NaN,NaN,NaN,NaN,NaN,NaN
is_canceled,119390.0,NaN,NaN,NaN,0.370416,0.0,0.0,0.0,1.0,1.0,0.482918
lead_time,119390.0,NaN,NaN,NaN,104.011416,0.0,18.0,69.0,160.0,737.0,106.863097
arrival_date_year,119390.0,NaN,NaN,NaN,2016.156554,2015.0,2016.0,2016.0,2017.0,2017.0,0.707476
arrival_date_month,119390,12,August,13877,NaN,NaN,NaN,NaN,NaN,NaN,NaN
arrival_date_week_number,119390.0,NaN,NaN,NaN,27.165173,1.0,16.0,28.0,38.0,53.0,13.605138
arrival_date_day_of_month,119390.0,NaN,NaN,NaN,15.798241,1.0,8.0,16.0,23.0,31.0,8.780829
stays_in_weekend_nights,119390.0,NaN,NaN,NaN,0.927599,0.0,0.0,1.0,2.0,19.0,0.998613
stays_in_week_nights,119390.0,NaN,NaN,NaN,2.500302,0.0,1.0,2.0,3.0,50.0,1.908286
adults,119390.0,NaN,NaN,NaN,1.856403,0.0,2.0,2.0,2.0,55.0,0.579261


In [163]:
# Calculate the number and percentage of missing values in each column
missing_values = pd.DataFrame({
    "missing_count": df.isnull().sum(),
    "missing_percentage": (df.isnull().sum() / len(df)) * 100
})

missing_values[missing_values["missing_count"] > 0].sort_values(
    by="missing_percentage",
    ascending=False
)

,missing_count,missing_percentage
company,112593,94.306893
agent,16340,13.686238
country,488,0.408744
children,4,0.003350


In [164]:
# Remove company because it contains more than 94% missing values
df = df.drop(columns="company")

print("Updated dataset shape:", df.shape)

Updated dataset shape: (119390, 31)


In [165]:
# Inspect records with missing children values
df.loc[
    df["children"].isna(),
    ["adults", "children", "babies", "customer_type"]
]

,adults,children,babies,customer_type
40600,2,NaN,0,Transient-Party
40667,2,NaN,0,Transient-Party
40679,3,NaN,0,Transient-Party
41160,2,NaN,0,Transient-Party


In [166]:
# Replace missing children values with 0
df["children"] = df["children"].fillna(0).astype(int)

In [167]:
# Inspect records with missing country
df.loc[
    df["country"].isna(),
    ["hotel", "market_segment", "distribution_channel", "customer_type"]
]

,hotel,market_segment,distribution_channel,customer_type
30,Resort Hotel,Direct,Direct,Transient
4127,Resort Hotel,Offline TA/TO,TA/TO,Transient
7092,Resort Hotel,Corporate,Corporate,Transient
7860,Resort Hotel,Direct,Direct,Transient
8779,Resort Hotel,Corporate,Corporate,Transient
...,...,...,...,...
65908,City Hotel,Complementary,Corporate,Transient
65909,City Hotel,Complementary,Corporate,Transient
65910,City Hotel,Complementary,Corporate,Transient
80830,City Hotel,Groups,TA/TO,Transient-Party


In [168]:
# Replace missing country values with Unknown
df["country"] = df["country"].fillna("Unknown")

In [169]:
# Investigate bookings where agent information is missing
df.loc[
    df["agent"].isna(),
    ["hotel", "market_segment", "distribution_channel", "customer_type"]
].value_counts()

hotel         market_segment  distribution_channel  customer_type  
Resort Hotel  Direct          Direct                Transient          3023
City Hotel    Direct          Direct                Transient          2103
              Corporate       Corporate             Transient          1685
Resort Hotel  Corporate       Corporate             Transient          1338
City Hotel    Groups          TA/TO                 Transient           967
                                                                       ... 
              Direct          TA/TO                 Transient-Party       1
              Complementary   Direct                Contract              1
              Direct          GDS                   Transient             1
              Groups          TA/TO                 Group                 1
              Online TA       GDS                   Transient             1
Name: count, Length: 90, dtype: int64

In [170]:
# Replace missing agent IDs with a separate category
# Agent is treated as a categorical identifier, not a numerical variable
df["agent"] = df["agent"].fillna("No Agent").astype(str)

In [171]:
# Confirm that no missing values remain
df.isna().sum()[df.isna().sum() > 0]

Series([], dtype: int64)

In [172]:
# Inspect invalid negative ADR values
negative_adr = df[df["adr"] < 0]

negative_adr[
    [
        "hotel",
        "is_canceled",
        "lead_time",
        "adults",
        "children",
        "babies",
        "stays_in_weekend_nights",
        "stays_in_week_nights",
        "adr",
        "reservation_status",
        "reservation_status_date"
    ]
]

,hotel,is_canceled,lead_time,adults,children,babies,stays_in_weekend_nights,stays_in_week_nights,adr,reservation_status,reservation_status_date
14969,Resort Hotel,0,195,2,0,0,4,6,-6.38,Check-Out,2017-03-15


In [173]:
# Remove records with invalid negative ADR
df = df[df["adr"] >= 0].copy()

# Validate cleaning
print("Negative ADR records remaining:", (df["adr"] < 0).sum())

Negative ADR records remaining: 0


In [174]:
# Expert numerical overview for anomaly detection
numeric_summary = df[
    [
        "lead_time",
        "stays_in_weekend_nights",
        "stays_in_week_nights",
        "adults",
        "children",
        "babies",
        "adr",
        "days_in_waiting_list",
        "required_car_parking_spaces",
        "total_of_special_requests"
    ]
].describe().T

numeric_summary

,count,mean,std,min,25%,50%,75%,max
lead_time,119389.0,104.010654,106.863220,0.0,18.00,69.00,160.0,737.0
stays_in_weekend_nights,119389.0,0.927573,0.998578,0.0,0.00,1.00,2.0,19.0
stays_in_week_nights,119389.0,2.500272,1.908267,0.0,1.00,2.00,3.0,50.0
adults,119389.0,1.856402,0.579263,0.0,2.00,2.00,2.0,55.0
children,119389.0,0.103887,0.398557,0.0,0.00,0.00,0.0,10.0
babies,119389.0,0.007949,0.097437,0.0,0.00,0.00,0.0,10.0
adr,119389.0,101.832028,50.535032,0.0,69.29,94.59,126.0,5400.0
days_in_waiting_list,119389.0,2.321169,17.594793,0.0,0.00,0.00,0.0,391.0
required_car_parking_spaces,119389.0,0.062518,0.245292,0.0,0.00,0.00,0.0,8.0
total_of_special_requests,119389.0,0.571368,0.792800,0.0,0.00,0.00,1.0,5.0


In [175]:
# Create derived variables for meaningful anomaly detection
df["total_guests"] = (
    df["adults"] +
    df["children"] +
    df["babies"]
)

df["total_nights"] = (
    df["stays_in_weekend_nights"] +
    df["stays_in_week_nights"]
)

In [176]:
# Inspect bookings with the highest number of guests
df.nlargest(
    20,
    "total_guests"
)[
    [
        "hotel",
        "adults",
        "children",
        "babies",
        "total_guests",
        "customer_type",
        "total_nights",
        "adr",
        "reservation_status"
    ]
]

,hotel,adults,children,babies,total_guests,customer_type,total_nights,adr,reservation_status
2173,Resort Hotel,55,0,0,55,Group,2,0.00,Canceled
1643,Resort Hotel,50,0,0,50,Group,3,0.00,Canceled
1539,Resort Hotel,40,0,0,40,Group,3,0.00,Canceled
1917,Resort Hotel,27,0,0,27,Group,4,0.00,Canceled
1962,Resort Hotel,27,0,0,27,Group,4,0.00,Canceled
1587,Resort Hotel,26,0,0,26,Group,7,0.00,Canceled
1752,Resort Hotel,26,0,0,26,Group,7,0.00,Canceled
1884,Resort Hotel,26,0,0,26,Group,7,0.00,Canceled
2003,Resort Hotel,26,0,0,26,Group,7,0.00,Canceled
2164,Resort Hotel,26,0,0,26,Group,7,0.00,Canceled


In [177]:
# Inspect bookings with the longest stays
df.nlargest(
    20,
    "total_nights"
)[
    [
        "hotel",
        "is_canceled",
        "lead_time",
        "total_guests",
        "total_nights",
        "customer_type",
        "adr",
        "reservation_status"
    ]
]

,hotel,is_canceled,lead_time,total_guests,total_nights,customer_type,adr,reservation_status
14038,Resort Hotel,0,126,1,69,Transient,110.00,Check-Out
14037,Resort Hotel,0,113,1,60,Transient,110.50,Check-Out
101794,City Hotel,0,140,0,57,Transient,8.34,Check-Out
9839,Resort Hotel,1,322,2,56,Transient,25.50,Canceled
33924,Resort Hotel,0,71,2,56,Transient,28.79,Check-Out
88017,City Hotel,0,16,0,49,Transient-Party,0.00,Check-Out
54704,City Hotel,0,206,2,48,Transient-Party,0.00,Check-Out
1655,Resort Hotel,0,30,2,46,Transient,0.00,Check-Out
32589,Resort Hotel,0,1,1,45,Transient,42.11,Check-Out
106561,City Hotel,0,11,0,43,Transient,0.00,Check-Out


In [178]:
# Check for logically invalid bookings
invalid_bookings = df[df["total_guests"] == 0]

print(f"Bookings with zero guests: {len(invalid_bookings)}")

invalid_bookings[
    [
        "hotel",
        "is_canceled",
        "lead_time",
        "adults",
        "children",
        "babies",
        "total_guests",
        "total_nights",
        "adr",
        "customer_type",
        "reservation_status"
    ]
]

Bookings with zero guests: 180


,hotel,is_canceled,lead_time,adults,children,babies,total_guests,total_nights,adr,customer_type,reservation_status
2224,Resort Hotel,0,1,0,0,0,0,3,0.00,Transient-Party,Check-Out
2409,Resort Hotel,0,0,0,0,0,0,0,0.00,Transient,Check-Out
3181,Resort Hotel,0,36,0,0,0,0,3,0.00,Transient-Party,Check-Out
3684,Resort Hotel,0,165,0,0,0,0,5,0.00,Transient-Party,Check-Out
3708,Resort Hotel,0,165,0,0,0,0,6,0.00,Transient-Party,Check-Out
...,...,...,...,...,...,...,...,...,...,...,...
115029,City Hotel,0,107,0,0,0,0,3,100.80,Transient,Check-Out
115091,City Hotel,0,1,0,0,0,0,1,0.00,Transient,Check-Out
116251,City Hotel,0,44,0,0,0,0,2,73.80,Transient,Check-Out
116534,City Hotel,0,2,0,0,0,0,7,22.86,Transient-Party,Check-Out


In [179]:
# Remove logically invalid bookings with no guests
df = df[df["total_guests"] > 0].copy()

print("Dataset shape after removing zero-guest bookings:", df.shape)

Dataset shape after removing zero-guest bookings: (119209, 33)


In [180]:
# Select important columns for easier duplicate inspection
cols_to_check = [
    "hotel",
    "is_canceled",
    "lead_time",
    "arrival_date_year",
    "arrival_date_month",
    "arrival_date_day_of_month",
    "adults",
    "children",
    "babies",
    "adr",
    "reservation_status"
]

# Display rows that are duplicates
df[df.duplicated(keep=False)][cols_to_check].head(15)

,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_day_of_month,adults,children,babies,adr,reservation_status
4,Resort Hotel,0,14,2015,July,1,2,0,0,98.00,Check-Out
5,Resort Hotel,0,14,2015,July,1,2,0,0,98.00,Check-Out
21,Resort Hotel,0,72,2015,July,1,2,0,0,84.67,Check-Out
22,Resort Hotel,0,72,2015,July,1,2,0,0,84.67,Check-Out
39,Resort Hotel,0,70,2015,July,2,2,0,0,137.00,Check-Out
43,Resort Hotel,0,70,2015,July,2,2,0,0,137.00,Check-Out
132,Resort Hotel,1,5,2015,July,5,2,0,0,97.00,Canceled
138,Resort Hotel,1,5,2015,July,5,2,0,0,97.00,Canceled
198,Resort Hotel,0,0,2015,July,7,1,0,0,109.80,Check-Out
200,Resort Hotel,0,0,2015,July,7,1,0,0,109.80,Check-Out


In [181]:
# Expert summary of exact duplicate records
duplicate_mask = df.duplicated(keep=False)
extra_duplicates = df.duplicated().sum()

print(f"Total rows before duplicate removal: {len(df):,}")
print(f"Rows involved in duplicate groups: {duplicate_mask.sum():,}")
print(f"Extra exact duplicate rows: {extra_duplicates:,}")
print(f"Exact duplicate percentage: {extra_duplicates / len(df) * 100:.2f}%")

Total rows before duplicate removal: 119,209
Rows involved in duplicate groups: 40,152
Extra exact duplicate rows: 31,987
Exact duplicate percentage: 26.83%


In [182]:
# Validate engineered features

guest_check = (
    df["total_guests"] ==
    df["adults"] +
    df["children"] +
    df["babies"]
)

night_check = (
    df["total_nights"] ==
    df["stays_in_weekend_nights"] +
    df["stays_in_week_nights"]
)

print("Incorrect total_guests rows:", (~guest_check).sum())
print("Incorrect total_nights rows:", (~night_check).sum())

Incorrect total_guests rows: 0
Incorrect total_nights rows: 0


In [183]:
# Remove temporary columns created during data validation
df.drop(
    columns=["total_guests", "total_nights"],
    inplace=True
)

# Check the remaining columns
df.columns.tolist()

['hotel',
 'is_canceled',
 'lead_time',
 'arrival_date_year',
 'arrival_date_month',
 'arrival_date_week_number',
 'arrival_date_day_of_month',
 'stays_in_weekend_nights',
 'stays_in_week_nights',
 'adults',
 'children',
 'babies',
 'meal',
 'country',
 'market_segment',
 'distribution_channel',
 'is_repeated_guest',
 'previous_cancellations',
 'previous_bookings_not_canceled',
 'reserved_room_type',
 'assigned_room_type',
 'booking_changes',
 'deposit_type',
 'agent',
 'days_in_waiting_list',
 'customer_type',
 'adr',
 'required_car_parking_spaces',
 'total_of_special_requests',
 'reservation_status',
 'reservation_status_date']

In [184]:
# Final validation before saving the cleaned dataset

print("Dataset shape:", df.shape)
print("Total missing values:", df.isnull().sum().sum())
print("Exact duplicate rows:", df.duplicated().sum())
print("Negative ADR values:", (df["adr"] < 0).sum())

# Check data types
print("\nData types:")
print(df.dtypes)

Dataset shape: (119209, 31)
Total missing values: 0
Exact duplicate rows: 31987
Negative ADR values: 0

Data types:
hotel                                        str
is_canceled                                int64
lead_time                                  int64
arrival_date_year                          int64
arrival_date_month                           str
arrival_date_week_number                   int64
arrival_date_day_of_month                  int64
stays_in_weekend_nights                    int64
stays_in_week_nights                       int64
adults                                     int64
children                                   int64
babies                                     int64
meal                                         str
country                                      str
market_segment                               str
distribution_channel                         str
is_repeated_guest                          int64
previous_cancellations                     int64
pr

In [185]:
# Save the final cleaned dataset
df.to_csv(
    "../datasets/Cleaned_hotel_bookings_cleaned.csv",
    index=False
)

print("Final cleaned dataset saved successfully.")
print("Final shape:", df.shape)

Final cleaned dataset saved successfully.
Final shape: (119209, 31)
